In [6]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

df = pd.read_csv("IMDB Dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
# Remove duplicates
df.drop_duplicates(inplace=True)
print(f"After removing duplicates: {df.shape}")

# --- IMPROVED PREPROCESSING ---
# Keep negation words (not, no, never) — important for sentiment!
# They are removed by default stopwords, so we keep them.
stop_words = set(stopwords.words("english"))
negation_words = {"not", "no", "never", "nor", "neither", "without",
                  "hardly", "barely", "scarcely", "don't", "doesn't",
                  "didn't", "won't", "wouldn't", "shouldn't", "couldn't",
                  "isn't", "wasn't", "aren't", "weren't", "haven't",
                  "hasn't", "hadn't", "can't", "cannot"}
stop_words = stop_words - negation_words  # Keep negation words!

def clean_text(text):
    text = text.lower()                          # Lowercase
    text = re.sub(r'<.*?>', ' ', text)           # Remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)          # Remove punctuation
    text = re.sub(r'\d+', '', text)              # Remove numbers
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return tokens

df["review"] = df["review"].apply(clean_text)
print("Sample cleaned review:")
print(df["review"].iloc[0][:10])

After removing duplicates: (49582, 2)
Sample cleaned review:
['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly']


In [8]:
X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify = balanced split!
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"Class balance (train):\n{y_train.value_counts()}")

Train size: 39665
Test size:  9917
Class balance (train):
sentiment
positive    19907
negative    19758
Name: count, dtype: int64


In [9]:
import gensim.downloader as api

# Load GloVe 300-dim (upgrade from 100-dim)
print("Loading GloVe 300-dim embeddings...")
glove_model = api.load("glove-wiki-gigaword-300")
print(f"Vocabulary size: {len(glove_model)}")
print(f"Embedding dimension: {glove_model.vector_size}")

Loading GloVe 300-dim embeddings...
Vocabulary size: 400000
Embedding dimension: 300


In [10]:
EMBED_DIM = 300  # Upgraded from 100

def sentence_vector(sentence):
    """Convert token list to mean GloVe vector.
    Returns zero vector if no tokens are found in vocabulary.
    """
    vectors = [glove_model[word] for word in sentence if word in glove_model]
    if len(vectors) == 0:
        return np.zeros(EMBED_DIM)
    return np.mean(vectors, axis=0)

print("Converting training set to vectors...")
X_train_vectors = np.array(X_train.apply(sentence_vector).tolist())

print("Converting test set to vectors...")
X_test_vectors = np.array(X_test.apply(sentence_vector).tolist())

print(f"X_train_vectors shape: {X_train_vectors.shape}")
print(f"X_test_vectors shape:  {X_test_vectors.shape}")

Converting training set to vectors...
Converting test set to vectors...
X_train_vectors shape: (39665, 300)
X_test_vectors shape:  (9917, 300)


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SentimentNN(nn.Module):
    def __init__(self, input_dim=300):
        super(SentimentNN, self).__init__()

        self.network = nn.Sequential(
            # طبقة واحدة مخفية بس
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

In [13]:
# Convert to tensors
X_train_tensor = torch.tensor(X_train_vectors, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_vectors,  dtype=torch.float32)

y_train_num = y_train.map({'positive': 1, 'negative': 0})
y_test_num  = y_test.map({'positive': 1, 'negative': 0})

y_train_tensor = torch.tensor(y_train_num.values, dtype=torch.float32).view(-1, 1)
y_test_tensor  = torch.tensor(y_test_num.values,  dtype=torch.float32).view(-1, 1)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=128, shuffle=True)
model = SentimentNN(input_dim=X_train_tensor.shape[1])

# Loss, Optimizer, Scheduler
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("Setup complete. Starting training...\n")

Setup complete. Starting training...



In [ ]:
num_epochs = 20
best_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    total_loss = 0
    correct = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += ((outputs >= 0.5).float() == batch_y).sum().item()

    avg_loss = total_loss / len(train_loader)
    train_acc = correct / len(X_train_tensor) * 100

    # ---- Validation (quick check on test set) ----
    model.eval()
    with torch.no_grad():
        val_preds = model(X_test_tensor)
        val_loss  = criterion(val_preds, y_test_tensor).item()
        val_acc   = ((val_preds >= 0.5).float() == y_test_tensor).float().mean().item() * 100

    scheduler.step(val_loss)  # Adjust LR based on validation loss

    # Save best model
    if val_loss < best_loss:
        best_loss = val_loss
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:>3}/{num_epochs}] "
              f"| Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% "
              f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# Restore best weights
model.load_state_dict(best_model_state)
print("\nTraining complete. Best model restored.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
with torch.no_grad():
    y_pred_probs = model(X_test_tensor)
    y_pred = (y_pred_probs >= 0.5).float()

y_true_np = y_test_tensor.numpy()
y_pred_np = y_pred.numpy()

accuracy = accuracy_score(y_true_np, y_pred_np)
print(f"\n{'='*45}")
print(f"  FINAL TEST ACCURACY: {accuracy * 100:.2f}%")
print(f"{'='*45}\n")

print("Classification Report:")
print(classification_report(y_true_np, y_pred_np, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_np, y_pred_np)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['Actual Neg', 'Actual Pos'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()